# Laplacian, diffusion and graph neural network

This notebook accompanies Exercises 10.1, 10.2 and 10.3. The Laplacian is the connecting theme: it measures signal variation, generates diffusion, transforms naturally when nodes are renumbered and supplies the smoothing operator of a graph convolution layer.

## Outline

1. [Laplacian and energy](#laplacien)
2. [Spectrum and diffusion](#spectre)
3. [Permuting nodes](#permutation)
4. [An initial node classification problem](#gcn)

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

<a id="laplacien"></a>
## 1. Laplacian and energy

Consider the path graph $1-2-3-4$ with unit weights. For a signal $u$, verify

$$
u^{\mathsf T}Lu
=
\frac12\sum_{i,j=1}^4 A_{ij}(u_i-u_j)^2.
$$

In [ ]:
n = 4
A = jnp.zeros((n, n))
A = A.at[jnp.arange(n - 1), jnp.arange(1, n)].set(1.0)
A = A + A.T
u = jnp.array([1.0, -1.0, 2.0, 0.5])

print("A =")
print(A)
print("u =", u)

Construct $D$ and $L=D-A$. Compute the energy in matrix form, then as a sum over edges. Also verify that $L\mathbf 1=0$.

In [ ]:
# Complete this section.

<a id="spectre"></a>
## 2. Spectrum and diffusion

Diagonalise $L$. Plot its eigenvectors as signals on the nodes, then compute

$$
u(t)=e^{-tL}u(0).
$$

Then add an edge of weight $0.2$ between the end nodes and observe the change in $\lambda_2$.

In [ ]:
# Complete this section.

<a id="permutation"></a>
## 3. Permuting nodes

The matrix $Q$ below changes the numbering. Construct

$$
A'=QAQ^{\mathsf T},\qquad
u'=Qu,
$$

recompute $L'$ from $A'$ and verify

$$
L'=QLQ^{\mathsf T},
\qquad
{u'}^{\mathsf T}L'u'=u^{\mathsf T}Lu.
$$

In [ ]:
permutation = jnp.array([2, 0, 3, 1])
Q = jnp.eye(n)[permutation]
print("Q =")
print(Q)

In [ ]:
# Complete this section.

<a id="gcn"></a>
## 4. An initial node classification problem

The following graph has two communities of eight nodes. The two classes correspond to the communities. The first feature at a node is informative, but several values have been perturbed enough to lie on the wrong side of zero.

We compare an affine classifier applied directly to $H$ with the same classifier applied after smoothing

$$
\widehat A H,
\qquad
\widehat A=\widetilde D^{-1/2}\widetilde A\widetilde D^{-1/2}.
$$

In [ ]:
n_communaute = 8
n = 2 * n_communaute
A = np.zeros((n, n))
for debut in (0, n_communaute):
    indices = np.arange(debut, debut + n_communaute)
    A[np.ix_(indices, indices)] = 1.0
    A[indices, indices] = 0.0
A[7, 8] = A[8, 7] = 0.15
A = jnp.asarray(A)

z = jnp.concatenate(
    [jnp.zeros(n_communaute, dtype=jnp.int32),
     jnp.ones(n_communaute, dtype=jnp.int32)]
)
signal = jnp.array(
    [-1.4, -0.8, 0.6, -1.1, 0.4, -0.9, -1.2, 0.5,
      1.2,  0.7, -0.5, 1.1, -0.3, 0.8, 1.3, -0.4]
)
H = jnp.column_stack([signal, jnp.ones(n)])
indices_apprentissage = jnp.array([0, 1, 8, 9])

print("labelled nodes:", indices_apprentissage)
print("labels:", z[indices_apprentissage])

Construct $L$, then $\widetilde A$, $\widetilde D$, $\widetilde L_{\rm sym}$ and $\widehat A$. Verify $\widehat A=I-\widetilde L_{\rm sym}$ and plot the first feature before and after smoothing.

In [ ]:
# Complete this section.

Now learn the parameters $W,b$ of the affine score $XW+b$ from the four labelled nodes. Do this once with $X=H$, then with $X=\widehat A H$. Compare predictions over all nodes.

In [ ]:
# Complete this section.

### Further questions

1. Why is smoothing useful here?
2. What happens if the weight of the edge between communities increases?
3. Apply $\widehat A$ repeatedly and observe over-smoothing.
4. Permute the nodes and verify that predictions are permuted in the same way.

## Summary

The Laplacian translates graph geometry into a positive semidefinite operator. Its energy measures variation along edges, its spectrum orders modes by regularity and its exponential generates diffusion. A convolution layer uses the same geometry to propagate features before learning a transformation. Equivariance ensures that the result does not depend on node numbering.